# 06 — SCL Join: NRC VAD v2 + SCL-OPP + SCL-NMA

Full outer join of three sentiment lexicons on `term`.

| source | entries | notes |
|--------|---------|-------|
| NRC VAD v2.1 | 54,801 | unigrams + bigrams + 2 trigrams; valence/arousal/dominance |
| SCL-OPP | 1,178 | opposing-polarity phrases; valence + POS pattern + tweet freq |
| SCL-NMA | 3,207 | negator/modal/adverb phrases; valence only |

**Output columns:** `term`, `ngrams`, `in_vad`, `in_opp`, `in_nma`,
`vad_valence`, `vad_arousal`, `vad_dominance`,
`opp_valence`, `opp_pos`, `opp_freq`, `opp_is_hashtag`, `nma_valence`,
`negators_contained`, `modals_contained`, `adverbs_contained`

After building the join, we audit whether constituent tokens of bigrams/trigrams
appear as unigrams, and whether sub-bigrams of trigrams appear as bigrams.

In [1]:
from pathlib import Path
import pandas as pd

DATA = Path('data')

VAD_FILE  = DATA / 'NRC-VAD-Lexicon-v2.1/NRC-VAD-Lexicon-v2.1.txt'
OPP_FILE  = DATA / 'SCL-OPP/SCL-OPP/SCL-OPP.txt'
NMA_FILE  = DATA / 'SCL-NMA/SCL-NMA/SCL-NMA.txt'
NEG_FILE  = DATA / 'SCL-NMA/SCL-NMA/list-English-negators.txt'
MOD_FILE  = DATA / 'SCL-NMA/SCL-NMA/list-English-modals.txt'
ADV_FILE  = DATA / 'SCL-NMA/SCL-NMA/list-English-adverbs.txt'

## Load each lexicon

In [2]:
# NRC VAD v2.1 — has header row
vad = pd.read_csv(VAD_FILE, sep='\t')
vad.columns = ['term', 'vad_valence', 'vad_arousal', 'vad_dominance']
print(f'VAD: {len(vad):,} entries')
vad.head(3)

VAD: 54,801 entries


,term,vad_valence,vad_arousal,vad_dominance
0,a battery,0.134,-0.298,-0.096
1,a bit,-0.096,-0.264,-0.214
2,a bunch,0.088,-0.350,-0.068


In [3]:
# SCL-OPP — 4 columns, no header
opp = pd.read_csv(OPP_FILE, sep='\t', header=None,
                  names=['term', 'opp_valence', 'opp_pos', 'opp_freq'])

# Flag hashtag terms (term contains '#') then strip all '#' characters
opp['opp_is_hashtag'] = opp['term'].str.contains('#', regex=False)
opp['term'] = opp['term'].str.replace('#', '', regex=False).str.strip()

print(f'SCL-OPP: {len(opp):,} entries ({opp["opp_is_hashtag"].sum()} hashtag terms stripped)')
opp[opp['opp_is_hashtag']].head(5)

SCL-OPP: 1,178 entries (12 hashtag terms stripped)


,term,opp_valence,opp_pos,opp_freq,opp_is_hashtag
9,smile,0.953,#,57000,True
64,leave a smile,0.766,V+D+N,37,True
304,finallyinthehotel exhausted,0.297,#+#,37,True
352,wantit,0.234,#,44,True
377,dog,0.219,#,8388,True


In [4]:
# SCL-NMA — 2 columns, no header
nma = pd.read_csv(NMA_FILE, sep='\t', header=None,
                  names=['term', 'nma_valence'])
print(f'SCL-NMA: {len(nma):,} entries')
nma.head(3)

SCL-NMA: 3,207 entries


,term,nma_valence
0,enthusiastic,0.986
1,excellence,0.984
2,greatness,0.972


## Full outer join

In [5]:
joined = (
    vad
    .merge(opp, on='term', how='outer')
    .merge(nma, on='term', how='outer')
)

# ngrams: word count of term
joined['ngrams'] = joined['term'].str.split().str.len()

# membership flags — which lexicon(s) each term appears in
joined['in_vad'] = joined['vad_valence'].notna()
joined['in_opp'] = joined['opp_valence'].notna()
joined['in_nma'] = joined['nma_valence'].notna()

# reorder columns
joined = joined[[
    'term', 'ngrams',
    'in_vad', 'in_opp', 'in_nma',
    'vad_valence', 'vad_arousal', 'vad_dominance',
    'opp_valence', 'opp_pos', 'opp_freq', 'opp_is_hashtag',
    'nma_valence',
]]

print(f'Joined: {len(joined):,} rows')
print(f'\nLexicon membership:')
print(f'  in_vad only:       {(joined["in_vad"] & ~joined["in_opp"] & ~joined["in_nma"]).sum():,}')
print(f'  in_opp only:       {(~joined["in_vad"] & joined["in_opp"] & ~joined["in_nma"]).sum():,}')
print(f'  in_nma only:       {(~joined["in_vad"] & ~joined["in_opp"] & joined["in_nma"]).sum():,}')
print(f'  in_vad + in_opp:   {(joined["in_vad"] & joined["in_opp"] & ~joined["in_nma"]).sum():,}')
print(f'  in_vad + in_nma:   {(joined["in_vad"] & ~joined["in_opp"] & joined["in_nma"]).sum():,}')
print(f'  in_opp + in_nma:   {(~joined["in_vad"] & joined["in_opp"] & joined["in_nma"]).sum():,}')
print(f'  in all three:      {(joined["in_vad"] & joined["in_opp"] & joined["in_nma"]).sum():,}')
joined.head(5)

Joined: 57,005 rows

Lexicon membership:
  in_vad only:       52,824
  in_opp only:       611
  in_nma only:       1,587
  in_vad + in_opp:   361
  in_vad + in_nma:   1,416
  in_opp + in_nma:   3
  in all three:      203


,term,ngrams,in_vad,in_opp,in_nma,vad_valence,vad_arousal,vad_dominance,opp_valence,opp_pos,opp_freq,opp_is_hashtag,nma_valence
0,3rd degree burns,3.0,False,True,False,NaN,NaN,NaN,-0.828,A+N+N,11.0,False,NaN
1,FALSE,1.0,False,False,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.444
2,TRUE,1.0,False,False,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.625
3,a bad deal,3.0,False,True,False,NaN,NaN,NaN,-0.594,D+A+N,13.0,False,NaN
4,a battery,2.0,True,False,False,0.134,-0.298,-0.096,NaN,NaN,NaN,NaN,NaN


## N-gram distribution by source

In [6]:
def ngram_counts(df, label):
    counts = df['ngrams'].value_counts().sort_index()
    counts.index = counts.index.map({1: 'unigram', 2: 'bigram', 3: 'trigram'}).fillna('4+')
    print(f'\n{label}')
    print(counts.to_string())

ngram_counts(joined[joined['in_vad']], 'NRC VAD v2')
ngram_counts(joined[joined['in_opp']], 'SCL-OPP')
ngram_counts(joined[joined['in_nma']], 'SCL-NMA')
ngram_counts(joined, 'Full join')


NRC VAD v2
ngrams
unigram    44730
bigram     10071
trigram        2

SCL-OPP
ngrams
unigram    602
bigram     311
trigram    265

SCL-NMA
ngrams
unigram    1623
bigram      922
trigram     605
4+           59

Full join
ngrams
unigram    44814
bigram     11259
trigram      872
4+            59


## Modifier flags: negators / modals / adverbs contained

In [7]:
negators = [l.strip() for l in NEG_FILE.read_text().splitlines() if l.strip()]
modals   = [l.strip() for l in MOD_FILE.read_text().splitlines() if l.strip()]
adverbs  = [l.strip() for l in ADV_FILE.read_text().splitlines() if l.strip()]

print(f'negators: {negators}')
print(f'modals:   {modals}')
print(f'adverbs:  {adverbs}')

def find_contained(term, word_list):
    """Return space-separated list of modifier tokens found in term (word-boundary match), or None."""
    tokens = str(term).lower().split()
    found = []
    for modifier in word_list:
        mod_tokens = modifier.split()
        for i in range(len(tokens) - len(mod_tokens) + 1):
            if tokens[i:i + len(mod_tokens)] == mod_tokens:
                found.append(modifier)
                break
    return ' '.join(found) if found else None

joined['negators_contained'] = joined['term'].apply(find_contained, word_list=negators)
joined['modals_contained']   = joined['term'].apply(find_contained, word_list=modals)
joined['adverbs_contained']  = joined['term'].apply(find_contained, word_list=adverbs)

print(f'\nTerms with negators: {joined["negators_contained"].notna().sum():,}')
print(f'Terms with modals:   {joined["modals_contained"].notna().sum():,}')
print(f'Terms with adverbs:  {joined["adverbs_contained"].notna().sum():,}')

negators: ['cannot', 'could not', 'did not', 'does not', 'had no', 'have no', 'may not', 'never', 'no', 'not', 'nothing', 'was no', 'was not', 'will not', 'would not']
modals:   ['can', 'could', 'may', 'might', 'must', 'should', 'would']
adverbs:  ['certainly', 'especially', 'extremely', 'fairly', 'highly', 'increasingly', 'less', 'more', 'most', 'much', 'much more', 'particularly', 'pretty', 'probably', 'quite', 'rather', 'really', 'relatively', 'so', 'too', 'very', 'very very']



Terms with negators: 547
Terms with modals:   515
Terms with adverbs:  880


In [8]:
print(f'Final shape: {joined.shape}')
print(f'Columns: {list(joined.columns)}')
joined.sample(10)

Final shape: (57005, 16)
Columns: ['term', 'ngrams', 'in_vad', 'in_opp', 'in_nma', 'vad_valence', 'vad_arousal', 'vad_dominance', 'opp_valence', 'opp_pos', 'opp_freq', 'opp_is_hashtag', 'nma_valence', 'negators_contained', 'modals_contained', 'adverbs_contained']


,term,ngrams,in_vad,in_opp,in_nma,vad_valence,vad_arousal,vad_dominance,opp_valence,opp_pos,opp_freq,opp_is_hashtag,nma_valence,negators_contained,modals_contained,adverbs_contained
43654,scribbled,1.0,True,False,False,-0.500,0.630,-0.375,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
42517,roadtrip,1.0,True,False,False,0.354,0.192,0.038,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23549,homology,1.0,True,False,False,0.142,-0.012,0.054,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19562,freebase,1.0,True,False,False,-0.708,1.000,-0.143,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22433,hastily,1.0,True,False,False,-0.458,0.218,0.218,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28079,largo,1.0,True,False,False,0.042,-0.040,0.034,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22189,handpick,1.0,True,False,False,0.667,0.619,0.593,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
40864,recondition,1.0,True,False,False,0.619,0.542,0.407,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10734,counterweigh,1.0,True,False,False,0.000,0.667,0.429,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7924,chock full,2.0,True,False,False,0.100,-0.106,-0.056,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Audit: constituent coverage

For every bigram and trigram in the joined dataset, check:
- Do its **unigram tokens** appear as entries in the dataset?
- For trigrams: do its **sub-bigrams** (token[0:2], token[1:3]) appear as bigram entries?

This tells us whether the lexicons are self-consistent — i.e. whether we can always
fall back to constituent lookups when a phrase is missing.

In [9]:
term_set = set(joined['term'].dropna().str.lower())

def check_constituents(term):
    tokens = term.lower().split()
    n = len(tokens)
    if n < 2:
        return None
    result = {'term': term, 'ngrams': n}
    result['unigram_tokens']   = tokens
    result['unigrams_in_set']  = [t for t in tokens if t in term_set]
    result['unigrams_missing'] = [t for t in tokens if t not in term_set]
    if n == 3:
        sub_bigrams = [' '.join(tokens[0:2]), ' '.join(tokens[1:3])]
        result['sub_bigrams']         = sub_bigrams
        result['sub_bigrams_in_set']  = [b for b in sub_bigrams if b in term_set]
        result['sub_bigrams_missing'] = [b for b in sub_bigrams if b not in term_set]
    return result

phrases = joined[joined['ngrams'] >= 2]['term'].dropna().tolist()
audit_rows = [check_constituents(t) for t in phrases]
audit = pd.DataFrame(audit_rows)

print(f'Phrases audited: {len(audit):,}')
audit['all_unigrams_covered'] = audit['unigrams_missing'].apply(lambda x: len(x) == 0)
print(f'\nUnigram coverage:')
print(audit['all_unigrams_covered'].value_counts().to_string())

trigram_audit = audit[audit['ngrams'] == 3].copy()
if len(trigram_audit):
    trigram_audit['all_sub_bigrams_covered'] = trigram_audit['sub_bigrams_missing'].apply(lambda x: len(x) == 0)
    print(f'\nTrigram sub-bigram coverage:')
    print(trigram_audit['all_sub_bigrams_covered'].value_counts().to_string())

Phrases audited: 12,190

Unigram coverage:
all_unigrams_covered
True     10085
False     2105

Trigram sub-bigram coverage:
all_sub_bigrams_covered
False    837
True      35


In [10]:
missing = audit[~audit['all_unigrams_covered']]
print(f'Phrases with at least one unigram token NOT in the dataset: {len(missing):,}')
missing[['term', 'ngrams', 'unigrams_missing']].head(20)

Phrases with at least one unigram token NOT in the dataset: 2,105


,term,ngrams,unigrams_missing
0,3rd degree burns,3,[3rd]
1,a bad deal,3,[a]
2,a battery,2,[a]
3,a bit,2,[a]
4,a bunch,2,[a]
5,a cappella,2,"[a, cappella]"
6,a couple,2,[a]
7,a crazy year,3,[a]
8,a drive,2,[a]
9,a few,2,[a]


In [11]:
if len(trigram_audit):
    missing_bigrams = trigram_audit[trigram_audit['sub_bigrams_missing'].apply(len) > 0]
    print(f'Trigrams with missing sub-bigrams: {len(missing_bigrams):,}')
    missing_bigrams[['term', 'sub_bigrams', 'sub_bigrams_missing']].head(20)

Trigrams with missing sub-bigrams: 837
